In [1]:
# Видаляємо стару копію репозиторію (якщо вже клонували раніше)
!rm -rf deep

# Клонуємо особистий репозиторій з конспектами і матеріалами
!git clone https://github.com/PetraStill/deep.git deep

# Переходимо в директорію репозиторію
%cd deep

Cloning into 'deep'...
remote: Enumerating objects: 9944, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9944 (delta 1), reused 0 (delta 0), pack-reused 9938 (from 5)
Receiving objects: 100% (9944/9944), 630.46 MiB | 29.72 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Updating files: 100% (10043/10043), done.
/content/deep


In [2]:
# Встановлюємо бібліотеку для завантаження датасетів з Kaggle
!pip install kagglehub

# Встановлюємо бібліотеку для NLP: токенізація, стоп-слова, стемінг
!pip install nltk

# Встановлюємо spaCy та мовну модель для англійської (для лематизації)
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# Встановлюємо kagglehub і завантажуємо датасет з Kaggle
!pip install -q kagglehub
import kagglehub, shutil, os, glob

# Завантажуємо датасет (потрібен Kaggle-акаунт, авторизація відбувається автоматично)
path = kagglehub.dataset_download("arhamrumi/amazon-product-reviews")

# Дивимось, що завантажилось
print(glob.glob(path + '/*'))

# Кладемо файл туди, де його очікує ноутбук
os.makedirs('../data', exist_ok=True)
csv_src = glob.glob(path + '/*.csv')[0]
shutil.copy(csv_src, '../data/Module_5_Lecture_1_Class_amazon_product_reviews.csv')
print('Готово!')

100%|██████████| 115M/115M [00:01<00:00, 117MB/s]

Extracting files...


['/root/.cache/kagglehub/datasets/arhamrumi/amazon-product-reviews/versions/1/Reviews.csv']
Готово!


In [4]:
# Завантажуємо всі необхідні бібліотеки
import re           # Регулярні вирази
import string       # Рядкові константи (рядки пунктуації)
import itertools    # Ітератори
from collections import Counter  # Підрахунок частот

import pandas as pd     # Датафрейми
import numpy as np      # Числові операції

import nltk             # NLTK = Natural Language Toolkit — бібліотека для NLP
nltk.download('stopwords')  # Завантажуємо список стоп-слів (одноразово)

from nltk.corpus import stopwords          # Список стоп-слів
from nltk.tokenize import word_tokenize   # Токенізатор
from nltk.stem.porter import PorterStemmer  # Стемер Портера

import spacy  # Бібліотека для NLP з лематизацією

from sklearn.metrics import roc_auc_score                        # Метрика AUC
from sklearn.feature_extraction.text import CountVectorizer      # BoW-векторизатор
from sklearn.feature_extraction.text import TfidfVectorizer      # TF-IDF-векторизатор
from sklearn.linear_model import LogisticRegression             # Класифікатор

from tqdm.auto import tqdm    # Прогрес-бар
tqdm.pandas()                 # Підключаємо до pandas

import matplotlib.pyplot as plt  # Графіки
import seaborn as sns            # Статистична візуалізація

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
# Зчитуємо набір даних; Id стає індексом датафрейму
# '../data/' — бо ноутбук лежить у папці notebooks/, а CSV — на рівень вище у data/
df = pd.read_csv('../data/Module_5_Lecture_1_Class_amazon_product_reviews.csv', index_col='Id')

In [6]:
# Переглядаємо перші рядки для розуміння структури даних
df.head(3)

,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
Id,,,,,,,,,
1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [7]:
# Переглядаємо базову інформацію про датасет (типи, кількість ненульових значень)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 568454 entries, 1 to 568454
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   ProductId               568454 non-null  object
 1   UserId                  568454 non-null  object
 2   ProfileName             568428 non-null  object
 3   HelpfulnessNumerator    568454 non-null  int64 
 4   HelpfulnessDenominator  568454 non-null  int64 
 5   Score                   568454 non-null  int64 
 6   Time                    568454 non-null  int64 
 7   Summary                 568427 non-null  object
 8   Text                    568454 non-null  object
dtypes: int64(4), object(5)
memory usage: 43.4+ MB


In [8]:
# Видаляємо нейтральні відгуки (Score = 3) — вони не мають чіткого сентименту
df = df.loc[df['Score'] != 3]

# Перевіряємо розмір після фільтрації
df.shape

(525814, 9)

In [9]:
# Створюємо цільову змінну:
# 1 (позитивний) — якщо Score = 4 або 5
# 0 (негативний) — якщо Score = 1 або 2
df['sentiment'] = [1 if score in [4, 5] else 0 for score in df['Score']]

In [10]:
# Підраховуємо кількість повністю ідентичних рядків (усі 10 колонок однакові)
df.duplicated().sum()

np.int64(255)

In [11]:
# Видаляємо повні дублікати і скидаємо індекс
df = df.drop_duplicates().reset_index(drop=True)

In [12]:
# Шукаємо однакові відгуки для різних версій продукту
df.groupby(['UserId', 'Time', 'Text']).count().sort_values('ProductId', ascending=False).head(10)

ProductId  \
UserId         Time       Text                                                            
A3TVZM3ZIXG8YW 1291420800 This review will make me sound really stupid, b...        182   
A36JDIN9RAAIEC 1292976000 I have two cats, one 6 and one 2 years old. Bot...         51   
A1TMAVN4CEM8U8 1336348800 Diamond Almonds<br />Almonds are a good source ...         43   
A1UQBFCERIP7VJ 1321401600 Stash Chamomile Herbal Tea is tea bags with dri...         38   
A29JUMRL1US6YP 1278201600 The pet food industry can be one of the most in...         35   
                          The pet food industry can be one of the most in...         34   
A25C5MVVCIYT5D 1304726400 I understand all the complaints about Science D...         34   
A1BD342U8BF3UC 1314230400 Ok, sounds crazy i know but i heard about this ...         28   
A24PZR4W555WQI 1294617600 My dogs and I love this food. They never leave ...         28   
A3A1OA237FOZFK 1296950400 I was a little hesitant to try these, especiall...         28   

                                                                              ProfileName  \
UserId         Time       Text                                                              
A3TVZM3ZIXG8YW 1291420800 This review will make me sound really stupid, b...          182   
A36JDIN9RAAIEC 1292976000 I have two cats, one 6 and one 2 years old. Bot...           51   
A1TMAVN4CEM8U8 1336348800 Diamond Almonds<br />Almonds are a good source ...           43   
A1UQBFCERIP7VJ 1321401600 Stash Chamomile Herbal Tea is tea bags with dri...           38   
A29JUMRL1US6YP 1278201600 The pet food industry can be one of the most in...           35   
                          The pet food industry can be one of the most in...           34   
A25C5MVVCIYT5D 1304726400 I understand all the complaints about Science D...           34   
A1BD342U8BF3UC 1314230400 Ok, sounds crazy i know but i heard about this ...           28   
A24PZR4W555WQI 1294617600 My dogs and I love this food. They never leave ...           28   
A3A1OA237FOZFK 1296950400 I was a little hesitant to try these, especiall...           28   

                                                                              HelpfulnessNumerator  \
UserId         Time       Text                                                                       
A3TVZM3ZIXG8YW 1291420800 This review will make me sound really stupid, b...                   182   
A36JDIN9RAAIEC 1292976000 I have two cats, one 6 and one 2 years old. Bot...                    51   
A1TMAVN4CEM8U8 1336348800 Diamond Almonds<br />Almonds are a good source ...                    43   
A1UQBFCERIP7VJ 1321401600 Stash Chamomile Herbal Tea is tea bags with dri...                    38   
A29JUMRL1US6YP 1278201600 The pet food industry can be one of the most in...                    35   
                          The pet food industry can be one of the most in...                    34   
A25C5MVVCIYT5D 1304726400 I understand all the complaints about Science D...                    34   
A1BD342U8BF3UC 1314230400 Ok, sounds crazy i know but i heard about this ...                    28   
A24PZR4W555WQI 1294617600 My dogs and I love this food. They never leave ...                    28   
A3A1OA237FOZFK 1296950400 I was a little hesitant to try these, especiall...                    28   

                                                                              HelpfulnessDenominator  \
UserId         Time       Text                                                                         
A3TVZM3ZIXG8YW 1291420800 This review will make me sound really stupid, b...                     182   
A36JDIN9RAAIEC 1292976000 I have two cats, one 6 and one 2 years old. Bot...                      51   
A1TMAVN4CEM8U8 1336348800 Diamond Almonds<br />Almonds are a good source ...                      43   
A1UQBFCERIP7VJ 1321401600 Stash Chamomile Herbal Tea is tea bags with dri...                      38   
A29JUMRL1US6YP 1278

In [13]:
# Видаляємо дублікати за ключовою комбінацією: UserId + Time + Text
# Залишаємо лише один відгук від одного користувача в один момент часу
df = df.drop_duplicates(subset={"UserId", "Time", "Text"})

# Перевіряємо фінальний розмір датасету
df.shape

(364133, 10)

In [14]:
# Словник скорочень: ключ = скорочення, значення = повна форма
# Джерело: https://stackoverflow.com/questions/19790188
contractions = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he's": "he is",
    "how'd": "how did",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'll": "i will",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'll": "it will",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "must've": "must have",
    "mustn't": "must not",
    "needn't": "need not",
    "oughtn't": "ought not",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "that'd": "that would",
    "that's": "that is",
    "there'd": "there had",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'll": "we will",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "where'd": "where did",
    "where's": "where is",
    "who'll": "who will",
    "who's": "who is",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are"
}

In [15]:
# Завантажуємо стандартний список стоп-слів для англійської
# Використовуємо set() для швидкого пошуку: O(1) замість O(n)
stop_words = set(stopwords.words('english')).union({'also', 'would', 'much', 'many'})

In [18]:
# Список слів-заперечень, які НЕ треба видаляти як стоп-слова
negations = {
    'aren', "aren't", 'couldn', "couldn't",
    'didn', "didn't", 'doesn', "doesn't",
    'don', "don't", 'hadn', "hadn't",
    'hasn', "hasn't", 'haven', "haven't",
    'isn', "isn't", 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't",
    'no', 'nor', 'not',
    'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't",
    'won', "won't", 'wouldn', "wouldn't"
}

# Видаляємо слова-заперечення зі стоп-слів, щоб вони залишались у тексті
stop_words = stop_words.difference(negations)

In [19]:
# Ініціалізуємо стемер Портера (для стемінгу — опціонально)
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

# Завантажуємо пайплайн spaCy для англійської
# disable=['parser','ner'] — вимикаємо непотрібні компоненти для швидкості
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

In [20]:
# Завантажуємо пайплайн spaCy для англійської
# disable=['parser','ner'] — вимикаємо непотрібні компоненти для швидкості
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

In [22]:
def normalize_text(raw_review):
    """
    Нормалізуємо сирий текст відгуку:
    HTML → email → URL → lowercase → contractions → stop words
    → нелітерні символи → лематизація → зайві пробіли.
    Повертає: очищений рядок із пробільними токенами.
    """

    # Видаляємо HTML-теги: <br />, <p>, <strong> тощо
    text = re.sub("<[^>]*>", " ", raw_review)

    # Видаляємо email-адреси (user@domain.com)
    text = re.sub("\\S*@\\S*[\\s]+", " ", text)

    # Видаляємо URL (http:// і https://)
    text = re.sub("https?:\\/\\/.*?[\\s]+", " ", text)

    # Переводимо весь текст у нижній регістр і розбиваємо на слова
    text = text.lower().split()

    # Замінюємо скорочення на повні форми: "don't" → "do not"
    text = [contractions.get(word) if word in contractions else word
            for word in text]

    # Знову об'єднуємо і розбиваємо — після розширення скорочень
    # "do not" — два слова, тому потрібне повторне split()
    text = " ".join(text).split()

    # Видаляємо стоп-слова (окрім заперечень, які залишили вище)
    text = [word for word in text if word not in stop_words]

    # Об'єднуємо у рядок для наступних операцій
    text = " ".join(text)

    # Видаляємо все, крім латинських літер, апострофа і пробілу
    # Це також видаляє залишкові цифри і пунктуацію
    text = re.sub("[^a-zA-Z' ]", "", text)

    # Лематизуємо за допомогою spaCy
    # nlp(text) — аналізуємо текст, token.lemma_ — лема кожного токена
    # len(token.lemma_) > 1 — відфільтровуємо одиничні символи
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if len(token.lemma_) > 1])

    # Видаляємо зайві пробіли (можуть з'явитись після видалення слів)
    text = re.sub("[\\s]+", " ", text)

    # Повертаємо нормалізований рядок
    return text

In [23]:
# Тестуємо функцію на реальному прикладі з відгуку
text = 'On a quest for the perfedc1112t,,, !!!! <br />%%2%% popcorn to compliment the Whirley Pop. \
 Don\'t get older, I\'m beginning to appreciate the more "natural" popcorn varieties, \
 and I suppose that\'s what attracted me to the Arrowhead Mills Organic Yellow Popcorn.<br /> \
 <br />I\'m no "organic" food expert. I just wanted some good tasting popcorn. \
 And, I feel like that\'s what I got. Using the Whirley Pop, with a very small amount of oil, \
 I\'ve had great results.'

# Виводимо оригінал і нормалізований текст
print('Оригінальний текст:', text, '#' * 30, sep='\n\n')
print('\nНормалізований текст:', normalize_text(text), sep='\n\n')

Оригінальний текст:

On a quest for the perfedc1112t,,, !!!! <br />%%2%% popcorn to compliment the Whirley Pop.  Don't get older, I'm beginning to appreciate the more "natural" popcorn varieties,  and I suppose that's what attracted me to the Arrowhead Mills Organic Yellow Popcorn.<br />  <br />I'm no "organic" food expert. I just wanted some good tasting popcorn.  And, I feel like that's what I got. Using the Whirley Pop, with a very small amount of oil,  I've had great results.

##############################

Нормалізований текст:

quest perfedct popcorn compliment whirley pop not get old beginning appreciate natural popcorn variety suppose attract arrowhead mill organic yellow popcorn no organic food expert want good tasting popcorn and feel like get use whirley pop small amount oil great result


In [24]:
# Зменшуємо датасет до 5000 прикладів для демонстрації
# (2500 позитивних + 2500 негативних)
df = df.groupby('sentiment').sample(2500, random_state=42)

# Перевіряємо розмір
df.shape

(5000, 10)

In [25]:
# Застосовуємо нормалізацію до кожного відгуку
# progress_apply замість apply показує прогрес-бар (з tqdm)
df['text_normalized'] = df['Text'].progress_apply(normalize_text)

  0%|          | 0/5000 [00:00<?, ?it/s]

In [26]:
# Розбиваємо дані на навчальну (80%) і тестову (20%) вибірки
train_idxs = df.sample(frac=0.8, random_state=42).index
test_idxs = [idx for idx in df.index if idx not in train_idxs]

# Формуємо X і y для навчання і тестування
X_train = df.loc[train_idxs, 'text_normalized']
X_test = df.loc[test_idxs, 'text_normalized']

y_train = df.loc[train_idxs, 'sentiment']
y_test = df.loc[test_idxs, 'sentiment']

In [27]:
# Ініціалізуємо і навчаємо CountVectorizer на тренувальних даних
# fit() будує словник з усіх слів у X_train
vect = CountVectorizer().fit(X_train)

# Виводимо розмір словника
len(vect.vocabulary_)

12632

In [28]:
# Переглядаємо перші 5 токенів словника (у алфавітному порядку)
vect.get_feature_names_out()[:5]

array(['aa', 'ab', 'aback', 'abandon', 'abba'], dtype=object)

In [29]:
# Перетворюємо тренувальні тексти у document-term matrix (DTM)
X_train_vectorized = vect.transform(X_train)
print(X_train_vectorized.shape)  # (4000, 12650)
X_train_vectorized

(4000, 12632)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 138241 stored elements and shape (4000, 12632)>

In [30]:
# Ініціалізуємо логістичну регресію і навчаємо на DTM
model = LogisticRegression(random_state=42)
model.fit(X_train_vectorized, y_train)

LogisticRegression(random_state=42)

In [31]:
# Передбачаємо клас для тестових даних і рахуємо AUC
predictions = model.predict(vect.transform(X_test))
print('AUC: ', roc_auc_score(y_test, predictions))

AUC:  0.8340852130325815


In [32]:
# Навчаємо допоміжну функцію для швидкого порівняння конфігурацій
def get_preds(text_column, algorithm, ngrams=(1, 1)):
    """
    Будуємо вектора, навчаємо LogisticRegression і виводимо AUC.

    Параметри:
      text_column (str): назва колонки з текстом ('Text' або 'text_normalized')
      algorithm (str): 'cv' — CountVectorizer, 'tfidf' — TfidfVectorizer
      ngrams (tuple): діапазон n-грамів, наприклад (1,1) або (1,2)
    """
    # Розбиваємо на навчальну і тестову вибірки
    X_train = df.loc[train_idxs, text_column]
    X_test = df.loc[test_idxs, text_column]
    y_train = df.loc[train_idxs, 'sentiment']
    y_test = df.loc[test_idxs, 'sentiment']

    # Ініціалізуємо векторизатор залежно від вибраного алгоритму
    if algorithm == 'cv':
        vect = CountVectorizer(ngram_range=ngrams).fit(X_train)
    elif algorithm == 'tfidf':
        vect = TfidfVectorizer(ngram_range=ngrams).fit(X_train)
    else:
        raise ValueError('Вибери алгоритм: `cv` або `tfidf`')

    # Виводимо розмір словника
    print('Vocabulary length: ', len(vect.vocabulary_))

    # Перетворюємо документи у матрицю документ-термін
    X_train_vectorized = vect.transform(X_train)
    print('Document-term matrix shape:', X_train_vectorized.shape)

    # Навчаємо модель логістичної регресії
    model = LogisticRegression(random_state=42)
    model.fit(X_train_vectorized, y_train)

    # Передбачаємо і виводимо AUC
    predictions = model.predict(vect.transform(X_test))
    print('AUC: ', roc_auc_score(y_test, predictions))

In [33]:
# Перевіряємо TF-IDF на нормалізованих даних
get_preds('text_normalized', 'tfidf')

Vocabulary length:  12632
Document-term matrix shape: (4000, 12632)
AUC:  0.8437593984962407


In [34]:
# Перевіряємо TF-IDF на ненормалізованих даних
get_preds('Text', 'tfidf')

Vocabulary length:  14214
Document-term matrix shape: (4000, 14214)
AUC:  0.8407017543859648


In [35]:
get_preds('text_normalized', 'cv', (1, 2))    # AUC = 0.8636
get_preds('text_normalized', 'tfidf', (1, 2)) # AUC = 0.8483
get_preds('text_normalized', 'cv', (2, 2))    # AUC = 0.7765
get_preds('Text', 'cv', (2, 2))               # AUC = 0.8374
get_preds('Text', 'tfidf', (2, 2))            # AUC = 0.8363

Vocabulary length:  127843
Document-term matrix shape: (4000, 127843)
AUC:  0.8636591478696741
Vocabulary length:  127843
Document-term matrix shape: (4000, 127843)
AUC:  0.8483709273182958
Vocabulary length:  115211
Document-term matrix shape: (4000, 115211)
AUC:  0.7765413533834586
Vocabulary length:  132784
Document-term matrix shape: (4000, 132784)
AUC:  0.8374436090225565
Vocabulary length:  132784
Document-term matrix shape: (4000, 132784)
AUC:  0.8363408521303258
